# 04 — Gộp các miếng cùng đĩa/thớt thành 1 món: chạy Grounding DINO + vật chứa
Mỗi ảnh chạy 2 lượt: "food" (như cũ) và "plate. bowl. tray. cutting board." (vật chứa). Lưu cả hai để đánh giá trên máy
(`tools/eval_group.py`): gộp / không gộp, trên UEC mini (có box thật, 58 ảnh khay nhiều món) và Food-101 tiny (200 ảnh bước 02).
File nguồn: `support_bounding/kaggle/04_group/notebook.py`.

In [ ]:
# Môi trường (chỉ cần torch + transformers có sẵn của Kaggle)
import sys, torch, transformers, PIL
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| pillow", PIL.__version__)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Thư viện `foodbox`
Các ô dưới ghi code thư viện ra `/kaggle/working/foodbox/` (nguồn: `support_bounding/foodbox/`; sửa ở đó rồi build lại, không sửa trong notebook).

In [ ]:
# Tạo thư mục chứa thư viện
import os
os.makedirs('foodbox', exist_ok=True)

In [ ]:
%%writefile foodbox/__init__.py
"""foodbox: luồng auto bounding box cho món ăn.

Dùng như module (nhập 1 ảnh → ra bounding box):
    from foodbox import FoodBoxDetector
    det = FoodBoxDetector()
    boxes = det.detect("khay_com.jpg")
    det.detect_and_save("khay_com.jpg", "ketqua/")
hoặc lệnh: foodbox-detect khay_com.jpg --out ketqua/

Các module:
- api.py         FoodBoxDetector: 1 ảnh → box (+ ảnh có khung, ảnh cắt, JSON)
- cli.py         lệnh foodbox-detect
- data.py        dò cấu trúc thư mục, chọn tập con, đọc box thật UECFOOD-256 và chuyển sang YOLO
- detectors.py   Grounding DINO, OWLv2, YOLO-World và SAM 2.1 (lấy box từ mask)
- postprocess.py NMS, lọc diện tích, bỏ box ôm cả khay, gán class
- metrics.py     IoU, Precision, Recall, AP@0.5
- labels.py      ghi nhãn YOLO, classes.txt, data.yaml
- crop.py        cắt từng box thành ảnh riêng
- autolabel.py   gán nhãn cả bộ ảnh <món>/*.jpg (bước 02)
- viz.py         vẽ box
- run_detect.py  chạy model trên 1 GPU (notebook gọi 2 tiến trình cho 2 GPU)
"""
__version__ = "0.1.0"


def __getattr__(name):
    # Nạp FoodBoxDetector khi cần: "import foodbox" không kéo theo torch/transformers
    if name == "FoodBoxDetector":
        from .api import FoodBoxDetector
        return FoodBoxDetector
    raise AttributeError(name)

In [ ]:
%%writefile foodbox/api.py
"""Dùng như một module: nhập 1 ảnh → ra bounding box (+ ảnh có khung, ảnh cắt, JSON).

    from foodbox import FoodBoxDetector
    det = FoodBoxDetector()                         # tự dùng GPU nếu có, không thì CPU
    boxes = det.detect("khay_com.jpg")              # list box, sắp theo điểm giảm dần
    det.detect_and_save("khay_com.jpg", "ketqua/")  # ghi khay_com_bbox.jpg, khay_com_1.jpg..., khay_com.json

Cùng model và tham số với pipeline gán nhãn (bước 02, NOTES.md v2.1): Grounding DINO base, prompt "food",
box ≥ 0.35, box phụ ≥ 0.45 (box lớn nhất luôn giữ), NMS 0.5, bỏ box < 0.2% ảnh, tối đa 20 box. Chỉ trả box,
không đặt tên món.
"""
import json
import os
import time

from PIL import Image, ImageOps

from .crop import crop_box
from .postprocess import group_dishes, pick_main, postprocess
from .viz import color_for, draw_boxes

MODEL_ID = "IDEA-Research/grounding-dino-base"
# group=True: gộp các miếng cùng đĩa/thớt thành 1 món (chạy thêm 1 lượt tìm vật chứa, chậm gấp ~2; NOTES v5)
DEFAULTS = {"prompts": ["food"], "conf": 0.35, "other_conf": 0.45, "nms_iou": 0.5, "min_area": 0.002, "max_det": 20,
            "group": True}


def _to_image(image):
    """Đường dẫn hoặc ảnh PIL -> ảnh PIL RGB đã xoay theo EXIF (giống pipeline)."""
    if isinstance(image, Image.Image):
        return ImageOps.exif_transpose(image).convert("RGB")
    with Image.open(image) as im:
        return ImageOps.exif_transpose(im).convert("RGB")


class FoodBoxDetector:
    """Phát hiện vùng thức ăn trong ảnh. Nạp model 1 lần (lần đầu tự tải ~900 MB từ HuggingFace), gọi nhiều lần."""

    def __init__(self, device="auto", model_id=MODEL_ID, prompts=None, conf=None, other_conf=None, nms_iou=None,
                 min_area=None, max_det=None, group=None):
        import torch
        from .detectors import GroundingDINO
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        p = dict(DEFAULTS)
        for k, v in dict(prompts=prompts, conf=conf, other_conf=other_conf, nms_iou=nms_iou, min_area=min_area,
                         max_det=max_det, group=group).items():
            if v is not None:
                p[k] = list(v) if k == "prompts" else v
        self.params, self.device, self.model_id = p, device, model_id
        self._det = GroundingDINO(prompts=p["prompts"], device=device, model_id=model_id)

    def detect(self, image):
        """Trả list box {id, x1, y1, x2, y2, score, largest}, sắp theo điểm giảm dần (id 1, 2, ...).

        Toạ độ pixel trên ảnh đã xoay EXIF; largest = box lớn nhất (box pipeline coi là món chính).
        """
        img = _to_image(image)
        return self._boxes(img)[0]

    def _boxes(self, img):
        p = self.params
        t = time.time()
        raw_b, raw_s = self._det.predict(img)
        b, s = postprocess(raw_b, raw_s, img.width, img.height, conf=p["conf"], nms_iou=p["nms_iou"],
                           min_area=p["min_area"], max_det=p["max_det"])
        if p["group"]:  # lượt 2: vật chứa -> gộp các miếng cùng đĩa/thớt thành 1 box
            from .detectors import CONTAINER_PROMPTS
            cb, cs = self._det.predict(img, prompts=CONTAINER_PROMPTS)
            b, s = group_dishes(b, s, cb, cs, img.width, img.height, nms_iou=p["nms_iou"])
        main = pick_main(b, s, "largest")
        keep = [k for k in range(len(s)) if k == main or s[k] >= p["other_conf"]]
        out = [{"id": i, "x1": round(float(b[k][0]), 1), "y1": round(float(b[k][1]), 1),
                "x2": round(float(b[k][2]), 1), "y2": round(float(b[k][3]), 1),
                "score": round(float(s[k]), 4), "largest": k == main}
               for i, k in enumerate(keep, 1)]
        return out, time.time() - t

    def detect_and_save(self, image, out_dir, vis=True, crops=True, pad=0.0, name=None):
        """Chạy detect và ghi vào out_dir: <tên>_bbox.jpg (khung đánh số i), <tên>_i.jpg (cắt đúng box i, từ ảnh gốc),
        <tên>.json (toạ độ). pad > 0 thì nới ảnh cắt mỗi phía pad x cạnh box. Trả dict kết quả (như trong JSON)."""
        img = _to_image(image)
        if name is None:
            name = os.path.splitext(os.path.basename(image))[0] if isinstance(image, (str, os.PathLike)) else "image"
        boxes, sec = self._boxes(img)
        os.makedirs(out_dir, exist_ok=True)
        files = {}
        if vis:
            files["bbox"] = f"{name}_bbox.jpg"
            draw_boxes(img, [[x["x1"], x["y1"], x["x2"], x["y2"]] for x in boxes],
                       labels=[str(x["id"]) for x in boxes], colors=[color_for(i) for i in range(len(boxes))]
                       ).save(os.path.join(out_dir, files["bbox"]), quality=92)
        for x in boxes:
            if crops:
                c, _ = crop_box(img, (x["x1"], x["y1"], x["x2"], x["y2"]), pad)
                x["crop"] = f"{name}_{x['id']}.jpg"
                c.save(os.path.join(out_dir, x["crop"]), quality=95)
        result = {"image": str(image) if isinstance(image, (str, os.PathLike)) else name, "width": img.width,
                  "height": img.height, "n_boxes": len(boxes), "boxes": boxes, "files": files,
                  "model": self.model_id, "device": self.device, "params": self.params, "seconds": round(sec, 3)}
        with open(os.path.join(out_dir, f"{name}.json"), "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=1)
        return result

In [ ]:
%%writefile foodbox/autolabel.py
"""Biến kết quả detector thành bộ nhãn YOLO hoàn chỉnh: ảnh, nhãn, hình vẽ box, classes.txt, data.yaml, report/failed.csv."""
import csv
import os
import random
import shutil

import numpy as np

from . import labels, viz
from .crop import save_crops, write_crops_csv
from .data import exif_rotated, load_image
from .postprocess import assign_classes, group_dishes, postprocess

REPORT_FIELDS = ["image", "class", "n_boxes", "mean_conf", "max_conf", "main_conf", "needs_review", "reasons"]


def review_reasons(scores, n_boxes, main_area_frac, review_conf, max_boxes):
    """Lý do cần xem lại một ảnh (chuỗi rỗng = ổn)."""
    r = []
    if n_boxes == 0:
        r.append("no_box")
    else:
        if float(np.max(scores)) < review_conf:
            r.append("low_conf")
        if n_boxes > max_boxes:
            r.append("many_boxes")
        if main_area_frac > 0.9:
            r.append("box_near_full_image")
    return ";".join(r)


def box_colors(cls, main_id):
    """Mỗi box một màu (yêu cầu đề bài): box món chính màu đầu, các box food_other các màu tiếp theo,
    nên trong một ảnh không có hai box trùng màu."""
    order = sorted(range(len(cls)), key=lambda k: cls[k] != main_id)
    out = [None] * len(cls)
    for rank, k in enumerate(order):
        out[k] = viz.color_for(rank)
    return out


def build_dataset(items, results, out, conf, nms_iou=0.5, min_area=0.002, max_area=1.0, drop_container=False,
                  main_rule="largest", review_conf=None, max_boxes=4, val_frac=0.2, seed=0, max_det=20,
                  other_conf=None, crops=False, crop_pad=0.0, group=False):
    """items: list {key, class, path, stem}; results: {key: {w, h, boxes, scores[, error]}} từ run_detect.

    Ghi vào out/: images/<món>/, labels/<món>/, visualized/<món>/, classes.txt, data.yaml, train.txt, val.txt,
    report.csv (mọi ảnh), failed.csv (ảnh không có box hoặc lỗi). Ảnh không có box vẫn được chép (để sửa tay)
    nhưng không nằm trong train.txt/val.txt. Trả (report_rows, failed_rows).

    other_conf: ngưỡng riêng cho box food_other (None = conf). Box món chính được dùng ngưỡng thấp hơn vì đã biết
    ảnh có món đó (tên thư mục); box thêm cần điểm cao hơn mới giữ, để bớt nhận nhầm đồ trang trí (02 v2: 56 box food_other).
    crops: cắt thêm từng box thành ảnh riêng từ ảnh gốc (không có khung): out/crops/<món>/<ảnh>_<k>.jpg + crops/crops.csv;
    crop_pad: mặc định 0 = cắt đúng box model khoanh; > 0 thì nới mỗi phía crop_pad x cạnh box.
    group: gộp các miếng cùng đĩa/thớt thành 1 box (cần c_boxes/c_scores từ run_detect --containers; NOTES v5).
    """
    review_conf = conf + 0.1 if review_conf is None else review_conf
    names = labels.class_list(sorted({it["class"] for it in items}))
    cid = {n: i for i, n in enumerate(names)}
    report, failed, ok_images, crop_rows = [], [], [], []
    for it in items:
        r = results.get(it["key"])
        rel = f"{it['class']}/{it['stem']}"
        if r is None or "error" in r:
            failed.append({"image": rel, "class": it["class"], "reason": (r or {}).get("error", "missing")})
            continue
        boxes, scores = postprocess(r["boxes"], r["scores"], r["w"], r["h"], conf=conf, nms_iou=nms_iou,
                                    min_area=min_area, max_area=max_area, drop_container=drop_container,
                                    max_det=max_det)
        if group:
            boxes, scores = group_dishes(boxes, scores, r.get("c_boxes"), r.get("c_scores"), r["w"], r["h"],
                                         nms_iou=nms_iou)
        cls = assign_classes(boxes, scores, cid[it["class"]], cid[labels.OTHER_CLASS], main_rule)
        if other_conf is not None:
            keep = [k for k, c in enumerate(cls) if c != cid[labels.OTHER_CLASS] or scores[k] >= other_conf]
            boxes, scores, cls = boxes[keep], scores[keep], [cls[k] for k in keep]

        # Ảnh: chép nguyên bản; ảnh có tag EXIF xoay thì lưu bản đã xoay để khớp toạ độ nhãn trong mọi công cụ
        ext = os.path.splitext(it["path"])[1].lower()
        img_rel = f"images/{rel}{ext}"
        os.makedirs(os.path.join(out, "images", it["class"]), exist_ok=True)
        img = load_image(it["path"])
        if exif_rotated(it["path"]):
            img.save(os.path.join(out, img_rel), quality=95)
        else:
            shutil.copy2(it["path"], os.path.join(out, img_rel))
        labels.write_yolo_txt(os.path.join(out, "labels", rel + ".txt"), cls, boxes, r["w"], r["h"])
        vis = viz.draw_boxes(img, boxes, labels=[f"{names[c]} {s:.2f}" for c, s in zip(cls, scores)],
                             colors=box_colors(cls, cid[it["class"]]))
        if crops and len(scores):
            crop_rows += save_crops(img, boxes, [names[c] for c in cls], os.path.join(out, "crops"), it["class"],
                                    it["stem"], img_rel, scores=scores,
                                    main_flags=[c == cid[it["class"]] for c in cls], pad=crop_pad)
        os.makedirs(os.path.join(out, "visualized", it["class"]), exist_ok=True)
        vis.save(os.path.join(out, "visualized", rel + ".jpg"), quality=85)

        main = [k for k, c in enumerate(cls) if c == cid[it["class"]]]
        main_frac = 0.0
        if main:
            b = boxes[main[0]]
            main_frac = (b[2] - b[0]) * (b[3] - b[1]) / (r["w"] * r["h"])
        reasons = review_reasons(scores, len(scores), main_frac, review_conf, max_boxes)
        report.append({"image": rel, "class": it["class"], "n_boxes": len(scores),
                       "mean_conf": round(float(np.mean(scores)), 4) if len(scores) else "",
                       "max_conf": round(float(np.max(scores)), 4) if len(scores) else "",
                       "main_conf": round(float(scores[main[0]]), 4) if main else "",
                       "needs_review": int(bool(reasons)), "reasons": reasons})
        if len(scores) == 0:  # ảnh có thức ăn mà không có box: không đưa vào train (YOLO sẽ hiểu là ảnh nền)
            failed.append({"image": rel, "class": it["class"], "reason": "no_box"})
        else:
            ok_images.append((it["class"], "./" + img_rel))

    # Chia train/val theo từng món (val_frac mỗi món), để train YOLO được ngay
    rng = random.Random(seed)
    train, val = [], []
    for c in sorted({c for c, _ in ok_images}):
        imgs = sorted(p for cc, p in ok_images if cc == c)
        rng.shuffle(imgs)
        n_val = max(1, round(len(imgs) * val_frac)) if len(imgs) > 1 else 0
        val += imgs[:n_val]
        train += imgs[n_val:]
    for fname, lst in [("train.txt", train), ("val.txt", val)]:
        with open(os.path.join(out, fname), "w", encoding="utf-8") as f:  # tên món có dấu tiếng Việt
            f.write("\n".join(sorted(lst)) + ("\n" if lst else ""))
    labels.write_classes(out, names)
    labels.write_data_yaml(out, names)
    with open(os.path.join(out, "classes.txt"), encoding="utf-8") as f:  # X-AnyLabeling cần classes.txt cạnh thư mục nhãn
        content = f.read()
    with open(os.path.join(out, "labels", "classes.txt"), "w", encoding="utf-8") as f:
        f.write(content)
    if crops:
        os.makedirs(os.path.join(out, "crops"), exist_ok=True)
        write_crops_csv(os.path.join(out, "crops"), crop_rows)
    _write_csv(os.path.join(out, "report.csv"), REPORT_FIELDS, report)
    _write_csv(os.path.join(out, "failed.csv"), ["image", "class", "reason"], failed)
    return report, failed


def _write_csv(path, fields, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

In [ ]:
%%writefile foodbox/cli.py
"""Lệnh foodbox-detect: nhập ảnh (hoặc thư mục ảnh) → ra bounding box.

  foodbox-detect khay_com.jpg --out ketqua/
  foodbox-detect anh1.jpg anh2.jpg thu_muc_anh/ --out ketqua/ --device cpu
  python -m foodbox.cli khay_com.jpg --out ketqua/          (khi chưa cài lệnh)

Mỗi ảnh ra: <tên>_bbox.jpg (khung đánh số 1, 2, ...), <tên>_1.jpg, <tên>_2.jpg ... (cắt đúng từng box), <tên>.json.
Ảnh trong thư mục con giữ nguyên cấu trúc thư mục con trong --out.
"""
import argparse
import json
import os
import sys
import time

from .data import is_image


def collect(inputs):
    """[(đường dẫn ảnh, thư mục con tương đối để ghi kết quả)] từ danh sách file / thư mục."""
    out = []
    for p in inputs:
        if os.path.isdir(p):
            for root, _, files in os.walk(p):
                rel = os.path.relpath(root, p)
                out += [(os.path.join(root, f), "" if rel == "." else rel) for f in sorted(files) if is_image(f)]
        elif os.path.isfile(p) and is_image(p):
            out.append((p, ""))
        else:
            print(f"[foodbox] bỏ qua (không phải ảnh / không tồn tại): {p}", flush=True)
    return out


def main(argv=None):
    if hasattr(sys.stdout, "reconfigure"):  # console Windows: in được tiếng Việt
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    ap = argparse.ArgumentParser(prog="foodbox-detect", description="Nhập ảnh món ăn → ra bounding box từng món")
    ap.add_argument("inputs", nargs="+", help="file ảnh và/hoặc thư mục ảnh")
    ap.add_argument("--out", default="foodbox_out", help="thư mục kết quả (mặc định foodbox_out/)")
    ap.add_argument("--device", default="auto", help="auto | cpu | cuda")
    ap.add_argument("--conf", type=float, default=None, help="ngưỡng box (mặc định 0.35)")
    ap.add_argument("--other-conf", type=float, default=None, help="ngưỡng box phụ (mặc định 0.45)")
    ap.add_argument("--prompt", action="append", default=None, help='câu chữ cho model (mặc định "food"); lặp lại để thêm')
    ap.add_argument("--no-crops", action="store_true", help="không cắt ảnh riêng từng box")
    ap.add_argument("--no-group", action="store_true",
                    help="không gộp các miếng cùng đĩa (mỗi miếng 1 khung; nhanh gấp ~2 vì bỏ lượt tìm đĩa)")
    ap.add_argument("--no-vis", action="store_true", help="không ghi ảnh có khung")
    ap.add_argument("--pad", type=float, default=0.0, help="nới ảnh cắt mỗi phía pad x cạnh box (mặc định 0 = đúng box)")
    a = ap.parse_args(argv)

    items = collect(a.inputs)
    if not items:
        print("[foodbox] không có ảnh nào để chạy")
        return 1
    from .api import FoodBoxDetector  # nạp torch/transformers chỉ khi thật sự chạy
    print("[foodbox] đang nạp model Grounding DINO (lần đầu tự tải ~900 MB từ HuggingFace)...", flush=True)
    t0 = time.time()
    det = FoodBoxDetector(device=a.device, prompts=a.prompt, conf=a.conf, other_conf=a.other_conf,
                          group=False if a.no_group else None)
    print(f"[foodbox] model sẵn sàng trên {det.device} sau {time.time() - t0:.0f} s; {len(items)} ảnh", flush=True)
    summary = []
    for i, (path, sub) in enumerate(items, 1):
        try:
            r = det.detect_and_save(path, os.path.join(a.out, sub), vis=not a.no_vis, crops=not a.no_crops, pad=a.pad)
            print(f"[foodbox] {i}/{len(items)} {path}: {r['n_boxes']} box ({r['seconds']:.1f} s)", flush=True)
            summary.append({"image": path, "n_boxes": r["n_boxes"], "out": os.path.join(a.out, sub)})
        except Exception as e:  # ảnh lỗi: báo rồi chạy tiếp
            print(f"[foodbox] {i}/{len(items)} {path}: LỖI {e!r}", flush=True)
            summary.append({"image": path, "error": repr(e)})
    os.makedirs(a.out, exist_ok=True)
    with open(os.path.join(a.out, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=1)
    print(f"[foodbox] xong: kết quả ở {a.out}/ ({time.time() - t0:.0f} s)")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile foodbox/crop.py
"""Cắt từng box thành file ảnh riêng: cắt ĐÚNG vùng box model khoanh được, từ ảnh gốc (không có khung vẽ)."""
import csv
import os
import shutil

# box_*: box model khoanh (toạ độ pixel thật); x1..y2: vùng đã cắt (= box làm tròn, kẹp trong ảnh; nới thêm nếu pad > 0)
CROP_FIELDS = ["crop", "image", "k", "class", "is_main", "score", "box_x1", "box_y1", "box_x2", "box_y2",
               "x1", "y1", "x2", "y2", "crop_w", "crop_h"]


def crop_box(img, box, pad=0.0):
    """Cắt 1 box (x1, y1, x2, y2 pixel). Mặc định pad=0: cắt đúng box; pad > 0 nới mỗi phía pad x cạnh box. Kẹp trong ảnh."""
    x1, y1, x2, y2 = [float(v) for v in box[:4]]
    dx, dy = (x2 - x1) * pad, (y2 - y1) * pad
    l, t = max(0, round(x1 - dx)), max(0, round(y1 - dy))
    r, b = min(img.width, round(x2 + dx)), min(img.height, round(y2 + dy))
    return img.crop((l, t, r, b)), (l, t, r, b)


def save_crops(img, boxes, class_names, out_dir, cls_folder, stem, image_rel, scores=None, main_flags=None,
               pad=0.0, min_size=16, quality=95):
    """Ghi <out_dir>/<cls_folder>/<stem>_<k>.jpg cho từng box; trả list dòng cho crops.csv.

    Box nhỏ hơn min_size pixel (cạnh ngắn) bị bỏ qua: ảnh cắt quá nhỏ không dùng được.
    """
    rows = []
    os.makedirs(os.path.join(out_dir, cls_folder), exist_ok=True)
    for k, b in enumerate(boxes):
        crop, (l, t, r, bt) = crop_box(img, b, pad)
        if min(crop.size) < min_size:
            continue
        name = f"{cls_folder}/{stem}_{k}.jpg"
        crop.save(os.path.join(out_dir, *name.split("/")), quality=quality)
        rows.append({"crop": name, "image": image_rel, "k": k, "class": class_names[k],
                     "is_main": int(main_flags[k]) if main_flags is not None else "",
                     "score": round(float(scores[k]), 4) if scores is not None else "",
                     "box_x1": round(float(b[0]), 1), "box_y1": round(float(b[1]), 1),
                     "box_x2": round(float(b[2]), 1), "box_y2": round(float(b[3]), 1),
                     "x1": l, "y1": t, "x2": r, "y2": bt, "crop_w": crop.width, "crop_h": crop.height})
    return rows


def write_crops_csv(out_dir, rows):
    with open(os.path.join(out_dir, "crops.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CROP_FIELDS)
        w.writeheader()
        w.writerows(rows)


def write_image_folder(src, crops, crops_dir, dest_dir, base):
    """Ghi bộ file của 1 ảnh vào dest_dir: <base>.<đuôi> (ảnh gốc, chép nguyên), <base>_bbox.jpg (khung đúng box model,
    đánh số i), <base>_<i>.jpg (ảnh cắt của khung i, i từ 1). crops: các dòng crops.csv của ảnh đó. Trả list dòng index."""
    from .data import load_image
    from .viz import color_for, draw_boxes
    crops = sorted(crops, key=lambda r: int(r["k"]))
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy2(src, os.path.join(dest_dir, base + os.path.splitext(src)[1].lower()))
    boxes = [[float(r[c]) for c in ("box_x1", "box_y1", "box_x2", "box_y2")] for r in crops]
    draw_boxes(load_image(src), boxes, labels=[str(i) for i in range(1, len(crops) + 1)],
               colors=[color_for(i) for i in range(len(crops))]).save(os.path.join(dest_dir, f"{base}_bbox.jpg"),
                                                                       quality=92)
    index = []
    for i, r in enumerate(crops, 1):
        name = f"{base}_{i}.jpg"
        shutil.copy2(os.path.join(crops_dir, *r["crop"].split("/")), os.path.join(dest_dir, name))
        index.append({"folder": os.path.basename(dest_dir), "file": name, "is_main": r["is_main"], "score": r["score"],
                      "x1": r["x1"], "y1": r["y1"], "x2": r["x2"], "y2": r["y2"]})
    return index

In [ ]:
%%writefile foodbox/data.py
"""Dò cấu trúc dataset, chọn tập con, đọc box thật của UECFOOD-256 (bb_info.txt) và chuyển sang YOLO."""
import os
import random
import re
from collections import deque

from PIL import Image, ImageOps

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SKIP_DIRS = {"__MACOSX", "__pycache__"}


def is_image(name):
    return os.path.splitext(name)[1].lower() in IMG_EXT


def _subdirs(d):
    try:
        return sorted(e.path for e in os.scandir(d)
                      if e.is_dir() and e.name not in SKIP_DIRS and not e.name.startswith("."))
    except OSError:
        return []


def _has_image(d, max_scan=200):
    # Chỉ xem tối đa max_scan mục đầu: thư mục món có hàng nghìn ảnh, không cần liệt kê hết
    try:
        with os.scandir(d) as it:
            for i, e in enumerate(it):
                if i >= max_scan:
                    break
                if e.is_file() and is_image(e.name):
                    return True
    except OSError:
        pass
    return False


def print_tree(root, max_depth=3, max_children=6):
    """In cây thư mục (chỉ thư mục, kèm số ảnh trực tiếp bên trong) để xem cấu trúc bản upload."""
    def walk(d, depth):
        subs = _subdirs(d)
        try:
            n_img = sum(1 for n in os.listdir(d) if is_image(n))
        except OSError:
            n_img = 0
        print("  " * depth + f"{os.path.basename(d) or d}/  ({len(subs)} thư mục con, {n_img} ảnh)")
        if depth >= max_depth:
            return
        for s in subs[:max_children]:
            walk(s, depth + 1)
        if len(subs) > max_children:
            print("  " * (depth + 1) + f"... còn {len(subs) - max_children} thư mục nữa")
    walk(root, 0)


def find_class_roots(root, max_depth=8, min_classes=2):
    """Tìm mọi thư mục dạng <thư_mục>/<tên_món>/*.jpg (vd. train/ và valid/ của Food-101 tiny).

    Không đi vào bên trong thư mục món (chứa ảnh) để không phải liệt kê hàng nghìn file.
    Kaggle gắn dataset ở /kaggle/input/datasets/<owner>/<slug>/..., nên thư mục món thường sâu 5-7 cấp
    (food101tiny: .../data/food-101-tiny/train/<món>, sâu 6) -> max_depth mặc định 8.
    """
    found = []
    queue = deque([(root, 0)])
    while queue:
        d, depth = queue.popleft()
        subs = _subdirs(d)
        with_img = [s for s in subs if _has_image(s)]
        if len(with_img) >= min_classes:
            found.append(d)
            continue
        if depth < max_depth:
            queue.extend((s, depth + 1) for s in subs if s not in with_img)
    return found


def list_class_images(class_roots):
    """Gộp ảnh của các thư mục gốc: {tên_món: [đường dẫn ảnh, ...]} (sắp xếp để chạy lại cho cùng kết quả)."""
    out = {}
    for r in class_roots:
        for s in _subdirs(r):
            imgs = sorted(os.path.join(s, n) for n in os.listdir(s) if is_image(n))
            if imgs:
                out.setdefault(os.path.basename(s), []).extend(imgs)
    return {k: sorted(v) for k, v in sorted(out.items())}


def sample_images(class_images, classes=None, n_classes=10, per_class=20, seed=0):
    """Chọn tập con: danh sách món (hoặc n_classes món ngẫu nhiên), mỗi món per_class ảnh (None = tất cả).

    Trả về list dict {key, class, path, stem}; key = "<món>/<tên file không đuôi>" dùng làm id ảnh.
    """
    rng = random.Random(seed)
    names = sorted(class_images)
    if classes is None:
        classes = names if n_classes is None or n_classes >= len(names) else sorted(rng.sample(names, n_classes))
    items = []
    for c in classes:
        imgs = class_images[c]
        pick = imgs if per_class is None or per_class >= len(imgs) else sorted(rng.sample(imgs, per_class))
        for p in pick:
            stem = os.path.splitext(os.path.basename(p))[0]
            items.append({"key": f"{c}/{stem}", "class": c, "path": p, "stem": stem})
    return items


def dedupe_keys(items):
    """Trùng tên file trong cùng món (vd. cùng tên ở train/ và valid/) -> thêm hậu tố _1, _2 vào stem và key."""
    seen = {}
    for it in items:
        n = seen.get(it["key"], 0)
        seen[it["key"]] = n + 1
        if n:
            it["stem"] += f"_{n}"
            it["key"] = f"{it['class']}/{it['stem']}"
    return items


def load_image(path):
    """Mở ảnh RGB, xoay theo EXIF (giống cv2.imread mà YOLO/Ultralytics và X-AnyLabeling dùng)."""
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert("RGB")


def exif_rotated(path):
    """True nếu ảnh có tag EXIF xoay (box thật có thể lệch nếu người gán nhãn dùng ảnh chưa xoay)."""
    try:
        with Image.open(path) as im:
            return im.getexif().get(0x0112, 1) not in (1, None)
    except OSError:
        return False


# ----------------------------- UECFOOD-256 -----------------------------

def safe_name(name):
    """Tên món -> tên thư mục/class an toàn: 'chicken-'n'-egg on rice' -> 'chicken-n-egg_on_rice'."""
    s = re.sub(r"[^\w\- ]+", "", name.strip().lower())
    return re.sub(r"\s+", "_", s)


def find_uec_root(root, max_depth=8):
    """Tìm thư mục chứa category.txt và các thư mục <id>/bb_info.txt."""
    queue = deque([(root, 0)])
    while queue:
        d, depth = queue.popleft()
        if os.path.isfile(os.path.join(d, "category.txt")) and os.path.isfile(os.path.join(d, "1", "bb_info.txt")):
            return d
        if depth < max_depth:
            queue.extend((s, depth + 1) for s in _subdirs(d) if not _has_image(s))
    return None


def read_uec_categories(uec_root):
    """category.txt ('id<TAB>name') -> {id: name}."""
    cats = {}
    with open(os.path.join(uec_root, "category.txt"), encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2 and parts[0].strip().isdigit():
                cats[int(parts[0])] = parts[1].strip()
    return cats


def read_uec_boxes(uec_root):
    """Đọc mọi <id>/bb_info.txt -> {img_id: [(cat_id, x1, y1, x2, y2), ...]}.

    Ảnh nhiều món nằm trong nhiều thư mục món, mỗi thư mục chỉ ghi box của món đó, nên phải gộp
    tất cả bb_info.txt mới có đủ box thật cho một ảnh.
    """
    boxes = {}
    for d in _subdirs(uec_root):
        cat = os.path.basename(d)
        f = os.path.join(d, "bb_info.txt")
        if not cat.isdigit() or not os.path.isfile(f):
            continue
        with open(f, encoding="utf-8") as fh:
            for line in fh:
                p = line.split()
                if len(p) != 5 or not p[0].isdigit():
                    continue  # bỏ dòng tiêu đề "img x1 y1 x2 y2"
                b = (int(cat), *(float(v) for v in p[1:]))
                if b not in boxes.setdefault(p[0], []):
                    boxes[p[0]].append(b)
    return boxes


def uec_class_images(uec_root, cat_ids, categories=None):
    """{tên_món_an_toàn: [ảnh trong thư mục <cat_id>]} cho các món được chọn."""
    categories = categories or read_uec_categories(uec_root)
    out = {}
    for cid in cat_ids:
        d = os.path.join(uec_root, str(cid))
        out[safe_name(categories[cid])] = sorted(os.path.join(d, n) for n in os.listdir(d) if is_image(n))
    return out


def clip_box(box, w, h):
    """Kẹp box (x1, y1, x2, y2) vào trong ảnh; trả None nếu box suy biến."""
    x1, y1, x2, y2 = box
    x1, x2 = sorted((min(max(x1, 0.0), w), min(max(x2, 0.0), w)))
    y1, y2 = sorted((min(max(y1, 0.0), h), min(max(y2, 0.0), h)))
    if x2 - x1 < 1 or y2 - y1 < 1:
        return None
    return (x1, y1, x2, y2)


def uec_gt_for_item(item, uec_boxes, cat_of_class, w, h):
    """Box thật của 1 ảnh: list (is_main, cat_id, (x1, y1, x2, y2)); is_main = box của món theo thư mục."""
    main_cat = cat_of_class[item["class"]]
    out = []
    for cat, *b in uec_boxes.get(item["stem"], []):
        cb = clip_box(b, w, h)
        if cb is not None:
            out.append((cat == main_cat, cat, cb))
    return out


def convert_uec_to_yolo(items, uec_boxes, cat_of_class, out_dir, categories=None):
    """Chuyển box thật UECFOOD-256 sang nhãn YOLO: <out_dir>/<món>/<ảnh>.txt + classes.txt.

    categories=None: class = các món trong tập con (theo thư mục) + 'food_other' cho box của món khác
    (cùng quy ước với nhãn tự động). categories={id: tên}: giữ đủ 256 class gốc (class_id = id - 1).
    Trả {key: [(is_main, cat_id, box), ...]} để đánh giá; ghi thêm it['w'], it['h'] vào từng item.
    """
    from .labels import OTHER_CLASS, class_list, write_classes, write_yolo_txt
    if categories is None:
        names = class_list(sorted({it["class"] for it in items}))
        cid = {n: i for i, n in enumerate(names)}
        to_id = lambda it, is_main, cat: cid[it["class"]] if is_main else cid[OTHER_CLASS]
    else:
        names = [categories[k] for k in sorted(categories)]
        to_id = lambda it, is_main, cat: cat - 1
    gts = {}
    for it in items:
        with Image.open(it["path"]) as im:
            w, h = ImageOps.exif_transpose(im).size
        it["w"], it["h"] = w, h
        g = uec_gt_for_item(it, uec_boxes, cat_of_class, w, h)
        gts[it["key"]] = g
        write_yolo_txt(os.path.join(out_dir, it["class"], it["stem"] + ".txt"),
                       [to_id(it, m, c) for m, c, _ in g], [b for *_, b in g], w, h)
    os.makedirs(out_dir, exist_ok=True)
    write_classes(out_dir, names)
    return gts

In [ ]:
%%writefile foodbox/detectors.py
"""Các model phát hiện theo câu chữ (open-vocabulary) và SAM 2.1 để lấy box ôm sát từ mask.

Mọi detector có cùng giao diện: predict(img PIL RGB) -> (boxes Nx4 pixel xyxy, scores N).
Ngưỡng ở đây để thấp (low_thr) để tính được AP; ngưỡng thật áp dụng ở bước hậu xử lý.
"""
import numpy as np
import torch

# Tên model -> id checkpoint mặc định
DEFAULT_IDS = {
    "gdino": "IDEA-Research/grounding-dino-base",   # Apache-2.0, HF transformers
    "owlv2": "google/owlv2-base-patch16-ensemble",  # Apache-2.0, HF transformers
    "yoloworld": "yolov8x-worldv2.pt",              # Ultralytics (AGPL-3.0)
    "sam": "sam2.1_b.pt",                           # SAM 2.1 base qua Ultralytics (trọng số Apache-2.0)
}


def _empty():
    return np.zeros((0, 4)), np.zeros(0)


def _topk(boxes, scores, k):
    boxes = np.asarray(boxes, float).reshape(-1, 4)
    scores = np.asarray(scores, float).reshape(-1)
    order = np.argsort(-scores, kind="stable")[:k]
    return boxes[order], scores[order]


class GroundingDINO:
    """Grounding DINO (Liu et al., ECCV 2024) qua HF transformers. Prompt dạng 'food.' (chữ thường, kết thúc bằng dấu chấm)."""

    def __init__(self, prompts=("food",), device="cuda", model_id=None, low_thr=0.05, max_det=100):
        from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
        mid = model_id or DEFAULT_IDS["gdino"]
        self.processor = AutoProcessor.from_pretrained(mid)
        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(mid).to(device).eval()
        self.prompts = list(prompts)
        self.text = " ".join(p.strip().lower().rstrip(".") + "." for p in prompts)
        self.device, self.low_thr, self.max_det = device, low_thr, max_det

    @torch.no_grad()
    def predict(self, img, prompts=None, extras=None):
        """prompts: đổi câu chữ cho ảnh này (vd. tên món); extras: vật phụ (vd. ['plate', 'bowl']) đưa vào câu chữ để
        model gán đĩa/bát cho chúng. Khi có prompts/extras, điểm của box = điểm cao nhất trên các token của prompts
        chính, và bỏ box có điểm token extras cao hơn (box đó là đĩa/bát chứ không phải thức ăn)."""
        if prompts is not None or extras:
            return self._predict_tokens(img, list(prompts or self.prompts), list(extras or []))
        inputs = self.processor(images=img, text=self.text, return_tensors="pt").to(self.device)
        out = self.model(**inputs)
        size = [(img.height, img.width)]
        post = self.processor.post_process_grounded_object_detection
        try:  # transformers >= 4.51 đổi box_threshold -> threshold
            r = post(out, inputs["input_ids"], threshold=self.low_thr, text_threshold=self.low_thr, target_sizes=size)[0]
        except TypeError:
            r = post(out, inputs["input_ids"], box_threshold=self.low_thr, text_threshold=self.low_thr, target_sizes=size)[0]
        if len(r["scores"]) == 0:
            return _empty()
        return _topk(r["boxes"].float().cpu().numpy(), r["scores"].float().cpu().numpy(), self.max_det)

    def _predict_tokens(self, img, prompts, extras):
        # Ghép câu chữ "p1. p2. plate. bowl." và nhớ vị trí ký tự của từng cụm để tìm token tương ứng
        phrases = [p.strip().lower().rstrip(".") for p in prompts + extras]
        text, spans = "", []
        for ph in phrases:
            spans.append((len(text), len(text) + len(ph)))
            text += ph + ". "
        text = text.strip()
        inputs = self.processor(images=img, text=text, return_tensors="pt").to(self.device)
        out = self.model(**inputs)
        offs = self.processor.tokenizer(text, return_offsets_mapping=True)["offset_mapping"]

        def tokens(sp):
            return [i for i, (a, b) in enumerate(offs) if b > a and a >= sp[0] and b <= sp[1]]

        main_idx = [i for sp in spans[:len(prompts)] for i in tokens(sp)]
        extra_idx = [i for sp in spans[len(prompts):] for i in tokens(sp)]
        probs = out.logits[0].float().sigmoid()  # (số query, số token)
        s_main = probs[:, main_idx].max(-1).values
        keep = s_main >= self.low_thr
        if extra_idx:
            keep &= s_main >= probs[:, extra_idx].max(-1).values
        if not bool(keep.any()):
            return _empty()
        cx, cy, w, h = out.pred_boxes[0][keep].float().unbind(-1)  # toạ độ chuẩn hoá (tâm, rộng, cao)
        boxes = torch.stack([(cx - w / 2) * img.width, (cy - h / 2) * img.height,
                             (cx + w / 2) * img.width, (cy + h / 2) * img.height], -1)
        return _topk(boxes.cpu().numpy(), s_main[keep].cpu().numpy(), self.max_det)


class OWLv2:
    """OWLv2 (Minderer et al., NeurIPS 2023) qua HF transformers. Mỗi prompt là một câu truy vấn."""

    def __init__(self, prompts=("food",), device="cuda", model_id=None, low_thr=0.02, max_det=100):
        from transformers import Owlv2ForObjectDetection, Owlv2Processor
        mid = model_id or DEFAULT_IDS["owlv2"]
        self.processor = Owlv2Processor.from_pretrained(mid)
        self.model = Owlv2ForObjectDetection.from_pretrained(mid).to(device).eval()
        self.queries = [list(prompts)]
        self.device, self.low_thr, self.max_det = device, low_thr, max_det

    @torch.no_grad()
    def predict(self, img):
        inputs = self.processor(text=self.queries, images=img, return_tensors="pt").to(self.device)
        out = self.model(**inputs)
        # OWLv2 đệm ảnh thành hình vuông (thêm phải/dưới) nên target size là cạnh dài x cạnh dài
        side = max(img.width, img.height)
        size = torch.tensor([[side, side]], device=self.device)
        try:
            r = self.processor.post_process_grounded_object_detection(out, threshold=self.low_thr, target_sizes=size)[0]
        except (AttributeError, TypeError):
            r = self.processor.post_process_object_detection(out, threshold=self.low_thr, target_sizes=size)[0]
        if len(r["scores"]) == 0:
            return _empty()
        b = r["boxes"].float().cpu().numpy()
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, img.width)
        b[:, [1, 3]] = b[:, [1, 3]].clip(0, img.height)
        return _topk(b, r["scores"].float().cpu().numpy(), self.max_det)


def _bgr(img):
    """PIL RGB -> numpy BGR (Ultralytics nhận mảng numpy theo thứ tự BGR như OpenCV)."""
    return np.ascontiguousarray(np.asarray(img)[:, :, ::-1])


class YOLOWorld:
    """YOLO-World v2 (Cheng et al., CVPR 2024) qua Ultralytics: rất nhanh, prompt = danh sách tên class."""

    def __init__(self, prompts=("food",), device="cuda", model_id=None, low_thr=0.01, max_det=100, imgsz=640):
        from ultralytics import YOLO
        self.model = YOLO(model_id or DEFAULT_IDS["yoloworld"])
        self.model.set_classes(list(prompts))
        self.device, self.low_thr, self.max_det, self.imgsz = device, low_thr, max_det, imgsz

    def predict(self, img):
        r = self.model.predict(_bgr(img), conf=self.low_thr, iou=0.7, imgsz=self.imgsz, max_det=self.max_det,
                               device=self.device, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            return _empty()
        return _topk(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), self.max_det)


class SAMRefiner:
    """Đưa box của detector vào SAM 2.1 làm prompt, lấy box bao quanh mask -> box ôm sát phần thức ăn.

    Box mới chỉ được co lại trong box gốc. Giữ box gốc khi mask rỗng hoặc quá nhỏ so với box (min_fill),
    vì khi đó SAM thường chỉ bắt được một mẩu nhỏ.
    """

    def __init__(self, device="cuda", model_id=None, min_fill=0.15):
        from ultralytics import SAM
        self.model = SAM(model_id or DEFAULT_IDS["sam"])
        self.device, self.min_fill = device, min_fill

    def refine(self, img, boxes):
        boxes = np.asarray(boxes, float).reshape(-1, 4)
        if len(boxes) == 0:
            return boxes.copy()
        r = self.model.predict(_bgr(img), bboxes=boxes.tolist(), device=self.device, verbose=False)[0]
        if r.masks is None:
            return boxes.copy()
        masks = r.masks.data.cpu().numpy() > 0.5
        out = boxes.copy()
        for k in range(min(len(masks), len(boxes))):
            m = masks[k]
            if m.shape != (img.height, img.width):  # phòng khi mask ở độ phân giải khác ảnh gốc
                sy, sx = m.shape[0] / img.height, m.shape[1] / img.width
            else:
                sy = sx = 1.0
            ys, xs = np.nonzero(m)
            if len(xs) == 0:
                continue
            nb = np.array([xs.min() / sx, ys.min() / sy, (xs.max() + 1) / sx, (ys.max() + 1) / sy])
            b = boxes[k]
            # Chỉ cho box co lại: mask tràn ra ngoài box gốc thường là đĩa/bàn bị dính vào
            nb = np.concatenate([np.maximum(nb[:2], b[:2]), np.minimum(nb[2:], b[2:])])
            if nb[2] - nb[0] < 1 or nb[3] - nb[1] < 1:
                continue
            box_area = max((b[2] - b[0]) * (b[3] - b[1]), 1e-9)
            if len(xs) / (sx * sy) >= self.min_fill * box_area:
                out[k] = nb
        return out


# Vật chứa để gộp các miếng cùng đĩa/thớt thành 1 món (postprocess.group_dishes, NOTES v5)
CONTAINER_PROMPTS = ["plate", "bowl", "tray", "cutting board"]


DETECTORS = {"gdino": GroundingDINO, "owlv2": OWLv2, "yoloworld": YOLOWorld}


def build_detector(name, prompts, device="cuda", model_id=None, low_thr=None):
    kw = {"prompts": prompts, "device": device, "model_id": model_id}
    if low_thr is not None:
        kw["low_thr"] = low_thr
    return DETECTORS[name](**kw)

In [ ]:
%%writefile foodbox/labels.py
"""Ghi nhãn YOLO (class_id x_center y_center width height, chuẩn hoá 0-1), classes.txt, data.yaml."""
import os

OTHER_CLASS = "food_other"


def xyxy_to_yolo(box, w, h):
    """(x1, y1, x2, y2) pixel -> (xc, yc, bw, bh) chuẩn hoá 0-1."""
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h)


def yolo_to_xyxy(yolo, w, h):
    xc, yc, bw, bh = yolo
    return ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)


def write_yolo_txt(path, class_ids, boxes, w, h):
    """Một dòng mỗi box. Ảnh không có box vẫn ghi file rỗng (YOLO hiểu là ảnh nền)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for c, b in zip(class_ids, boxes):
            f.write(f"{int(c)} " + " ".join(f"{min(max(v, 0.0), 1.0):.6f}" for v in xyxy_to_yolo(b, w, h)) + "\n")


def read_yolo_txt(path, w, h):
    """Đọc lại file YOLO -> (class_ids, boxes pixel)."""
    cls, boxes = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.split()
            if len(p) == 5:
                cls.append(int(p[0]))
                boxes.append(yolo_to_xyxy([float(v) for v in p[1:]], w, h))
    return cls, boxes


def class_list(main_classes):
    """Danh sách class: các món theo tên thư mục + 'food_other' ở cuối."""
    return sorted(main_classes) + [OTHER_CLASS]


def write_classes(out_dir, names):
    with open(os.path.join(out_dir, "classes.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(names) + "\n")


def write_data_yaml(out_dir, names, train="train.txt", val="val.txt"):
    """data.yaml cho Ultralytics: ảnh ở images/<món>/, nhãn ở labels/<món>/ (YOLO tự đổi images -> labels).

    Không ghi 'path': Ultralytics khi đó lấy thư mục chứa data.yaml, nên chép cả thư mục output đi đâu cũng train được.
    """
    lines = ["# Train: yolo detect train data=data.yaml model=yolo11n.pt", f"train: {train}", f"val: {val}",
             f"nc: {len(names)}", "names:"]
    lines += [f"  {i}: {n}" for i, n in enumerate(names)]
    with open(os.path.join(out_dir, "data.yaml"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

In [ ]:
%%writefile foodbox/metrics.py
"""Đo độ chính xác box (không phân biệt món): IoU, Precision, Recall, F1, AP@0.5."""
import numpy as np

from .postprocess import area, as_boxes, iou_matrix, pick_main


def match_image(boxes, scores, gt_boxes, iou_thr=0.5):
    """Ghép tham lam theo điểm giảm dần (kiểu VOC/COCO): mỗi box thật ghép tối đa 1 box dự đoán.

    Trả (tp, iou, gt_idx) theo đúng thứ tự box dự đoán đầu vào.
    """
    boxes, gt_boxes = as_boxes(boxes), as_boxes(gt_boxes)
    n = len(boxes)
    tp, best_iou, gt_idx = np.zeros(n, bool), np.zeros(n), np.full(n, -1)
    if n == 0 or len(gt_boxes) == 0:
        return tp, best_iou, gt_idx
    ious = iou_matrix(boxes, gt_boxes)
    used = np.zeros(len(gt_boxes), bool)
    for i in np.argsort(-np.asarray(scores), kind="stable"):
        cand = np.where(used, -1.0, ious[i])
        j = int(np.argmax(cand))
        best_iou[i] = ious[i].max()
        if cand[j] >= iou_thr:
            tp[i], gt_idx[i], used[j] = True, j, True
    return tp, best_iou, gt_idx


def average_precision(scores, tp, n_gt):
    """AP = diện tích dưới đường Precision-Recall (nội suy mọi điểm, như VOC2010+)."""
    if n_gt == 0 or len(scores) == 0:
        return 0.0
    order = np.argsort(-np.asarray(scores), kind="stable")
    tp = np.asarray(tp, float)[order]
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    rec = ctp / n_gt
    prec = ctp / np.maximum(ctp + cfp, 1e-9)
    mrec = np.concatenate([[0.0], rec, [rec[-1]]])
    mpre = np.concatenate([[1.0], prec, [0.0]])
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def evaluate(preds, gts, iou_thr=0.5, conf=None):
    """preds: {key: (boxes, scores)} (lọc ngưỡng thấp); gts: {key: [(is_main, cat, box), ...]}.

    - ap50: AP@0.5 trên toàn bộ box dự đoán (không phụ thuộc ngưỡng confidence).
    - P/R/F1, mean_iou, ...: tính ở ngưỡng `conf`; conf=None thì dùng ngưỡng cho F1 cao nhất (best_conf).
      Mỗi model có thang điểm khác nhau (OWLv2 điểm thấp, GDINO điểm cao) nên so ở ngưỡng tốt nhất của từng model.
    - mean_iou: mỗi box thật lấy IoU cao nhất với box dự đoán (0 nếu không có), rồi lấy trung bình.
    - area_ratio: trung vị (diện tích box dự đoán / box thật) của các cặp ghép đúng; > 1 = box rộng hơn box thật.
    - main_acc_*: tỉ lệ ảnh mà box được chọn làm món chính (lớn nhất / điểm cao nhất) trùng box món theo thư mục.
    """
    all_s, all_tp, all_iou, all_ratio, all_key = [], [], [], [], []
    n_gt = 0
    for key, gt in gts.items():
        gt_boxes = [g[2] for g in gt]
        boxes, scores = preds.get(key, (np.zeros((0, 4)), np.zeros(0)))
        boxes, scores = as_boxes(boxes), np.asarray(scores, float)
        tp, iou, gi = match_image(boxes, scores, gt_boxes, iou_thr)
        ratio = np.full(len(scores), np.nan)
        if len(gt_boxes):
            ga = area(gt_boxes)
            ratio[tp] = area(boxes[tp]) / np.maximum(ga[gi[tp]], 1e-9)
        n_gt += len(gt_boxes)
        all_s.append(scores), all_tp.append(tp), all_iou.append(iou), all_ratio.append(ratio)
        all_key += [key] * len(scores)
    s, tp = np.concatenate(all_s), np.concatenate(all_tp)
    ap = average_precision(s, tp, n_gt)

    if conf is None:  # ngưỡng cho F1 cao nhất
        order = np.argsort(-s, kind="stable")
        ctp = np.cumsum(tp[order])
        k = np.arange(1, len(s) + 1)
        f1 = 2 * ctp / np.maximum(k + n_gt, 1e-9)
        conf = float(s[order][int(np.argmax(f1))]) if len(s) else 0.0
    m = s >= conf
    n_tp, n_pred = int(tp[m].sum()), int(m.sum())
    prec = n_tp / n_pred if n_pred else 0.0
    rec = n_tp / n_gt if n_gt else 0.0
    ratio = np.concatenate(all_ratio)[m & tp]

    ious_gt, main_ok, n_main, no_box = [], {"largest": 0, "score": 0}, 0, 0
    for key, gt in gts.items():
        boxes, scores = preds.get(key, (np.zeros((0, 4)), np.zeros(0)))
        boxes, scores = as_boxes(boxes), np.asarray(scores, float)
        keep = scores >= conf
        boxes, scores = boxes[keep], scores[keep]
        no_box += int(len(scores) == 0)
        if gt:
            g = as_boxes([x[2] for x in gt])
            ious_gt += list(iou_matrix(g, boxes).max(axis=1)) if len(boxes) else [0.0] * len(g)
        mains = as_boxes([x[2] for x in gt if x[0]])
        if len(mains):
            n_main += 1
            for rule in main_ok:
                i = pick_main(boxes, scores, rule)
                if i >= 0 and iou_matrix(boxes[i:i + 1], mains).max() >= iou_thr:
                    main_ok[rule] += 1
    return {
        "ap50": ap, "conf": conf, "precision": prec, "recall": rec,
        "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
        "mean_iou": float(np.mean(ious_gt)) if ious_gt else 0.0,
        "area_ratio": float(np.median(ratio)) if len(ratio) else float("nan"),
        "main_acc_largest": main_ok["largest"] / n_main if n_main else float("nan"),
        "main_acc_score": main_ok["score"] / n_main if n_main else float("nan"),
        "n_images": len(gts), "n_gt": n_gt, "n_pred": n_pred, "n_tp": n_tp, "images_no_box": no_box,
    }

In [ ]:
%%writefile foodbox/postprocess.py
"""Hậu xử lý box: NMS (bỏ box trùng), lọc diện tích, bỏ box ôm cả khay, gán class."""
import numpy as np


def as_boxes(boxes):
    return np.asarray(boxes, dtype=np.float64).reshape(-1, 4)


def area(b):
    b = as_boxes(b)
    return np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)


def intersection(a, b):
    """Ma trận diện tích giao nhau |a| x |b|."""
    a, b = as_boxes(a), as_boxes(b)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    return wh[..., 0] * wh[..., 1]


def iou_matrix(a, b):
    inter = intersection(a, b)
    union = area(a)[:, None] + area(b)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0)


def nms(boxes, scores, iou_thr=0.5):
    """NMS không phân biệt class: giữ box điểm cao, bỏ box trùng IoU > iou_thr. Trả chỉ số giữ lại."""
    boxes, scores = as_boxes(boxes), np.asarray(scores, dtype=np.float64)
    order = np.argsort(-scores, kind="stable")
    keep = []
    while order.size:
        i = order[0]
        keep.append(int(i))
        if order.size == 1:
            break
        ious = iou_matrix(boxes[i:i + 1], boxes[order[1:]])[0]
        order = order[1:][ious <= iou_thr]
    return keep


def drop_containers(boxes, scores=None, contain_thr=0.85, min_children=2, min_rel_score=0.5):
    """Bỏ box chứa >= min_children box khác (box ôm cả khay/bàn chứa nhiều món). Trả chỉ số giữ lại.

    Box con "nằm trong" box cha nếu phần giao chiếm >= contain_thr diện tích box con. Chỉ tính box con có
    điểm >= min_rel_score x điểm box cha: bát phở 0.8 chứa vài box rác 0.1 (miếng thịt, hành) vẫn được giữ.
    """
    boxes = as_boxes(boxes)
    if len(boxes) < min_children + 1:
        return list(range(len(boxes)))
    scores = np.ones(len(boxes)) if scores is None else np.asarray(scores, dtype=np.float64)
    inter = intersection(boxes, boxes)
    inside = inter / np.maximum(area(boxes)[None, :], 1e-9) >= contain_thr  # inside[i, j]: j nằm trong i
    np.fill_diagonal(inside, False)
    bigger = area(boxes)[:, None] > area(boxes)[None, :]
    strong = scores[None, :] >= min_rel_score * scores[:, None]
    n_children = (inside & bigger & strong).sum(axis=1)
    return [i for i in range(len(boxes)) if n_children[i] < min_children]


def postprocess(boxes, scores, w, h, conf=0.0, nms_iou=0.5, min_area=0.002, max_area=1.0,
                drop_container=False, max_det=20):
    """Chuỗi lọc chuẩn; trả (boxes, scores) đã sắp theo điểm giảm dần.

    min_area / max_area là tỉ lệ diện tích so với ảnh (box quá nhỏ = nhiễu).
    Mặc định max_area=1.0 và drop_container=False: kết quả 01 (v1.1 trong NOTES.md) cho thấy max_area=0.98 xoá box
    đúng của ảnh chụp cận (món chiếm cả khung) và drop_container xoá món nhiều miếng (sushi, chả giò);
    tắt cả hai thì AP@0.5 tăng ~0.09. Box gần cả ảnh được đánh dấu xem lại trong report.csv thay vì xoá.
    """
    boxes, scores = as_boxes(boxes), np.asarray(scores, dtype=np.float64)
    frac = area(boxes) / float(w * h)
    m = (scores >= conf) & (frac >= min_area) & (frac <= max_area)
    boxes, scores = boxes[m], scores[m]
    keep = nms(boxes, scores, nms_iou)
    boxes, scores = boxes[keep], scores[keep]
    if drop_container:
        keep = drop_containers(boxes, scores)
        boxes, scores = boxes[keep], scores[keep]
    return boxes[:max_det], scores[:max_det]


def _inside(a, b):
    """inside[i, j] = tỉ lệ diện tích box a[i] nằm trong box b[j]."""
    return intersection(a, b) / np.maximum(area(a)[:, None], 1e-9)


def group_dishes(boxes, scores, c_boxes=None, c_scores=None, w=None, h=None, c_conf=0.30, inside_thr=0.7,
                 c_min_area=0.01, contain_thr=0.8, nms_iou=0.5, keep_parent=True):
    """Gộp các miếng của CÙNG MỘT đĩa/thớt thành 1 box (yêu cầu người dùng, NOTES v5). Trả (boxes, scores) sắp theo điểm.

    1. Box to bao trọn >= 2 box khác (mỗi box con nằm >= contain_thr trong nó): giữ box to, bỏ các box con.
    2. Vật chứa (c_boxes: plate/bowl/tray/cutting board, điểm >= c_conf, >= c_min_area ảnh): mỗi box thức ăn thuộc vật chứa
       NHỎ NHẤT bao nó (>= inside_thr diện tích box nằm trong). Các box cùng vật chứa -> 1 box = hợp của chúng (ôm sát
       phần thức ăn, không ôm cả đĩa), điểm = điểm cao nhất. Vật chứa bao >= 2 vật chứa khác (khay chứa nhiều bát) không
       dùng để gộp, để các món khác bát trong khay vẫn tách riêng.
    3. Box không thuộc vật chứa nào giữ nguyên. Cuối cùng NMS lần nữa (box hợp có thể trùng box to sẵn có).
    keep_parent=False: bỏ bước 1 (để đánh giá riêng từng bước).
    """
    boxes, scores = as_boxes(boxes), np.asarray(scores, dtype=np.float64)
    if len(boxes) == 0:
        return boxes, scores
    # 1) box to chứa >= 2 box con -> bỏ box con
    ins = _inside(boxes, boxes)  # ins[i, j]: i nằm trong j
    np.fill_diagonal(ins, 0)
    bigger = area(boxes)[None, :] > area(boxes)[:, None]
    is_parent = ((ins >= contain_thr) & bigger).sum(axis=0) >= 2
    child = ((ins >= contain_thr) & bigger & is_parent[None, :]).any(axis=1) & keep_parent
    boxes, scores = boxes[~child], scores[~child]

    # 2) gộp theo vật chứa nhỏ nhất
    if c_boxes is not None and len(c_boxes) and len(boxes) >= 2:
        cb, cs = as_boxes(c_boxes), np.asarray(c_scores, dtype=np.float64)
        keep = cs >= c_conf
        if w and h:
            keep &= area(cb) / float(w * h) >= c_min_area
        cb = cb[keep]
        if len(cb):
            cin = _inside(cb, cb)
            np.fill_diagonal(cin, 0)
            tray = ((cin >= inside_thr) & (area(cb)[None, :] > area(cb)[:, None])).sum(axis=0) >= 2
            cb = cb[~tray]
        if len(cb):
            fin = _inside(boxes, cb)  # box thức ăn i nằm trong vật chứa j
            owner = np.full(len(boxes), -1)
            for i in range(len(boxes)):
                cand = np.where(fin[i] >= inside_thr)[0]
                if len(cand):
                    owner[i] = cand[np.argmin(area(cb[cand]))]
            out_b, out_s = [], []
            for j in sorted(set(owner[owner >= 0])):
                g = np.where(owner == j)[0]
                gb = boxes[g]
                out_b.append([gb[:, 0].min(), gb[:, 1].min(), gb[:, 2].max(), gb[:, 3].max()])
                out_s.append(scores[g].max())
            for i in np.where(owner < 0)[0]:
                out_b.append(boxes[i].tolist())
                out_s.append(scores[i])
            boxes, scores = as_boxes(out_b), np.asarray(out_s, dtype=np.float64)

    # 3) NMS lần nữa, sắp theo điểm
    keep = nms(boxes, scores, nms_iou)
    return boxes[keep], scores[keep]


def pick_main(boxes, scores, rule="largest"):
    """Chỉ số box món chính: 'largest' (diện tích lớn nhất) hoặc 'score' (điểm cao nhất). -1 nếu không có box."""
    if len(scores) == 0:
        return -1
    if rule == "score":
        return int(np.argmax(scores))
    return int(np.argmax(area(boxes)))


def assign_classes(boxes, scores, main_id, other_id, rule="largest"):
    """Box món chính lấy class theo tên thư mục, các box còn lại là food_other."""
    i = pick_main(boxes, scores, rule)
    return [main_id if k == i else other_id for k in range(len(scores))]

In [ ]:
%%writefile foodbox/run_detect.py
"""Chạy detector (+ SAM) trên danh sách ảnh, trên 1 GPU. Notebook mở 2 tiến trình (mỗi GPU một tiến trình).

Ví dụ:
  CUDA_VISIBLE_DEVICES=0 python -m foodbox.run_detect --items items.json --models gdino --sam --out preds
  CUDA_VISIBLE_DEVICES=1 python -m foodbox.run_detect --items items.json --models gdino --shard 1 --nshards 2 ...

Kết quả: <out>/preds_<biến thể>_shard<i>.json, biến thể = tên model hoặc '<model>+sam' (hoặc '<model><tag>' khi
dùng --prompt-sets). Trong lúc chạy, cứ --ckpt-every ảnh lại ghi <out>/partial_<biến thể>_shard<i>.json; chạy lại cùng
lệnh thì bỏ qua ảnh đã xong (chạy tiếp). Kernel bị dừng giữa chừng thì load_preds vẫn đọc được phần đã lưu.
"""
import argparse
import glob
import json
import os
import time

import numpy as np
import torch

from .data import load_image
from .detectors import CONTAINER_PROMPTS, SAMRefiner, build_detector
from .postprocess import postprocess


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def _reset_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def _peak_mb():
    return torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else 0.0


def _save(path, variant, meta, images, quiet=False):
    tmp = path + ".tmp"  # ghi file tạm rồi đổi tên: bị dừng giữa lúc ghi cũng không hỏng file cũ
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump({"variant": variant, "meta": meta, "images": images}, f)
    os.replace(tmp, path)
    if not quiet:
        print(f"[run_detect] đã ghi {path} ({len(images)} ảnh)", flush=True)


def _resume(out, variant, shard):
    """Kết quả đã lưu theo đợt của lần chạy trước (nếu có): {key: kết quả}."""
    p = os.path.join(out, f"partial_{variant}_shard{shard}.json")
    if not os.path.exists(p):
        return {}
    with open(p, encoding="utf-8") as f:
        images = json.load(f)["images"]
    print(f"[run_detect] chạy tiếp {variant}: đã có {len(images)} ảnh trong {p}", flush=True)
    return images


def _finish(out, variant, shard, meta, images):
    _save(os.path.join(out, f"preds_{variant}_shard{shard}.json"), variant, meta, images)
    p = os.path.join(out, f"partial_{variant}_shard{shard}.json")
    if os.path.exists(p):
        os.remove(p)


def _item_prompts(ps, it):
    """Prompt cho 1 ảnh: cố định, hoặc theo tên món (thư mục) khi mode='class' ('french_toast' -> 'french toast')."""
    return [it["class"].replace("_", " ")] if ps.get("mode") == "class" else ps["prompts"]


def run(items, models, prompts, out, sam=False, sam_top=20, device="cuda", low_thr=None, shard=0, nshards=1,
        log_every=25, ckpt_every=50, prompt_sets=None, containers=False):
    """prompt_sets (chỉ Grounding DINO): list {tag, mode: fixed|class, prompts, extras}. Mỗi bộ là một biến thể
    '<model><tag>'; extras (vd. ['plate', 'bowl']) được thêm vào câu chữ rồi bỏ các box mang nhãn đó.
    containers (chỉ gdino): chạy thêm 1 lượt tìm vật chứa (CONTAINER_PROMPTS), lưu c_boxes/c_scores để gộp món."""
    os.makedirs(out, exist_ok=True)
    items = items[shard::nshards]
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    print(f"[run_detect] shard {shard}/{nshards}: {len(items)} ảnh, models={models}, sam={sam}, device={gpu}", flush=True)
    sets = prompt_sets or [{"tag": "", "mode": "fixed", "prompts": prompts, "extras": []}]
    refiner = None
    for name in models:
        if containers and name != "gdino":
            raise ValueError("containers chỉ hỗ trợ gdino")
        if prompt_sets and name != "gdino":
            raise ValueError("prompt_sets chỉ hỗ trợ gdino (prompt đổi theo từng ảnh)")
        # 1) Detector: nạp model, chạy thử 1 ảnh (khởi động CUDA), rồi đo thời gian từng ảnh
        _reset_peak()
        t0 = time.time()
        det = build_detector(name, prompts, device=device, low_thr=low_thr)
        load_s = time.time() - t0
        if items:
            det.predict(load_image(items[0]["path"]))
        for ps in sets:
            variant = name + ps["tag"]
            images = _resume(out, variant, shard)
            todo = [it for it in items if it["key"] not in images]
            for k, it in enumerate(todo):
                try:
                    img = load_image(it["path"])
                    _sync()
                    t = time.time()
                    if prompt_sets:
                        boxes, scores = det.predict(img, prompts=_item_prompts(ps, it), extras=ps.get("extras"))
                    else:
                        boxes, scores = det.predict(img)
                    r = {"w": img.width, "h": img.height, "boxes": boxes.round(2).tolist(),
                         "scores": scores.round(5).tolist()}
                    if containers:  # lượt 2: vật chứa (đĩa, bát, khay, thớt)
                        cb, cs = det.predict(img, prompts=CONTAINER_PROMPTS)
                        r.update(c_boxes=cb.round(2).tolist(), c_scores=cs.round(5).tolist())
                    _sync()
                    r["time_s"] = time.time() - t
                    images[it["key"]] = r
                except Exception as e:  # ảnh lỗi: ghi lại rồi chạy tiếp
                    images[it["key"]] = {"w": 0, "h": 0, "boxes": [], "scores": [], "time_s": 0.0, "error": repr(e)}
                    print(f"[run_detect] LỖI {it['key']}: {e!r}", flush=True)
                if (k + 1) % ckpt_every == 0:  # lưu theo đợt
                    _save(os.path.join(out, f"partial_{variant}_shard{shard}.json"), variant, {}, images, quiet=True)
                if (k + 1) % log_every == 0 or k + 1 == len(todo):
                    print(f"[run_detect] {variant}: {len(images)}/{len(items)}", flush=True)
            meta = {"model": variant, "prompts": ps["prompts"], "mode": ps.get("mode", "fixed"),
                    "extras": ps.get("extras", []), "load_s": load_s, "peak_vram_mb": _peak_mb(), "gpu": gpu}
            _finish(out, variant, shard, meta, images)

        # 2) SAM: lấy top sam_top box (sau NMS) của mỗi ảnh, thay bằng box bao quanh mask
        if sam:
            if refiner is None:
                refiner = SAMRefiner(device=device)
            _reset_peak()
            sam_images = _resume(out, f"{name}+sam", shard)
            todo = [it for it in items if it["key"] not in sam_images]
            for k, it in enumerate(todo):
                r = images[it["key"]]
                if "error" in r:
                    sam_images[it["key"]] = r
                    continue
                img = load_image(it["path"])
                b, s = postprocess(r["boxes"], r["scores"], r["w"], r["h"], conf=0.0, nms_iou=0.5,
                                   min_area=0.0, max_area=1.0, drop_container=False, max_det=sam_top)
                _sync()
                t = time.time()
                nb = refiner.refine(img, b)
                _sync()
                sam_images[it["key"]] = {"w": r["w"], "h": r["h"], "boxes": nb.round(2).tolist(),
                                         "scores": s.round(5).tolist(), "time_s": r["time_s"] + time.time() - t}
                if (k + 1) % ckpt_every == 0:
                    _save(os.path.join(out, f"partial_{name}+sam_shard{shard}.json"), f"{name}+sam", {}, sam_images,
                          quiet=True)
                if (k + 1) % log_every == 0 or k + 1 == len(todo):
                    print(f"[run_detect] {name}+sam: {len(sam_images)}/{len(items)}", flush=True)
            meta_sam = dict(meta, model=f"{name}+sam", peak_vram_mb=max(meta["peak_vram_mb"], _peak_mb()))
            _finish(out, f"{name}+sam", shard, meta_sam, sam_images)
        del det
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def load_preds(out):
    """Gộp các shard: {biến thể: {"meta": [meta mỗi shard], "images": {key: kết quả}}}."""
    res = {}
    finals = sorted(glob.glob(os.path.join(out, "preds_*_shard*.json")))
    done = {os.path.basename(p)[len("preds_"):] for p in finals}
    # Kết quả lưu theo đợt của shard chưa chạy xong (kernel bị dừng): vẫn đọc, meta ghi partial=True
    partials = [p for p in sorted(glob.glob(os.path.join(out, "partial_*_shard*.json")))
                if os.path.basename(p)[len("partial_"):] not in done]
    for p in finals + partials:
        with open(p, encoding="utf-8") as f:
            d = json.load(f)
        r = res.setdefault(d["variant"], {"meta": [], "images": {}})
        r["meta"].append(d["meta"] or {"partial": True, "peak_vram_mb": 0.0, "load_s": 0.0})
        r["images"].update(d["images"])
    return res


def as_arrays(r):
    """Kết quả 1 ảnh -> (boxes Nx4, scores N) numpy."""
    return np.asarray(r["boxes"], float).reshape(-1, 4), np.asarray(r["scores"], float)


def launch(jobs, timeout_min=40, poll_s=30, cwd=None):
    """Chạy song song các tiến trình run_detect, mỗi job trên 1 GPU: jobs = [(gpu_id, [đối số...], file_log)].

    In dòng log cuối của từng job mỗi poll_s giây; quá timeout_min thì dừng hẳn (tránh kernel treo mất quota).
    cwd: thư mục chạy tiến trình con. Ultralytics tải trọng số (.pt, CLIP) vào thư mục hiện tại, nên đặt ngoài
    /kaggle/working để output không phình thêm ~600 MB (lỗi của 01 v1). Đường dẫn trong args phải là tuyệt đối.
    Trả list mã thoát (None = bị dừng do quá giờ).
    """
    import subprocess
    import sys
    if cwd:
        os.makedirs(cwd, exist_ok=True)
    procs = []
    lib_parent = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))  # để tiến trình con import được foodbox
    for gpu, args, log in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8",
                   PYTHONPATH=os.pathsep.join(p for p in [lib_parent, os.environ.get("PYTHONPATH")] if p))
        fh = open(log, "w", encoding="utf-8")
        p = subprocess.Popen([sys.executable, "-m", "foodbox.run_detect", *args], stdout=fh,
                             stderr=subprocess.STDOUT, env=env, cwd=cwd, start_new_session=True)
        procs.append((p, fh, log, gpu))
        print(f"[launch] GPU {gpu}: {' '.join(args)}", flush=True)
    t0, codes = time.time(), [None] * len(procs)
    while any(p.poll() is None for p, *_ in procs):
        if time.time() - t0 > timeout_min * 60:
            print(f"[launch] quá {timeout_min} phút -> dừng các job còn chạy", flush=True)
            for p, *_ in procs:
                if p.poll() is None:
                    os.killpg(p.pid, 9)
            break
        time.sleep(poll_s)
        for p, _, log, gpu in procs:
            with open(log, encoding="utf-8", errors="replace") as f:
                lines = [l for l in f.read().splitlines() if l.strip()]
            print(f"[launch] {time.time() - t0:5.0f}s GPU {gpu}: {lines[-1] if lines else '(chưa có log)'}", flush=True)
    for k, (p, fh, log, gpu) in enumerate(procs):
        codes[k] = p.poll()
        fh.close()
        print(f"[launch] GPU {gpu} mã thoát {codes[k]} (log: {log})", flush=True)
    return codes


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--items", required=True, help="file JSON: list {key, path, ...}")
    ap.add_argument("--models", required=True, help="vd. gdino,owlv2,yoloworld")
    ap.add_argument("--prompts", default="food", help="các prompt cách nhau bởi dấu |, vd. 'food|dish of food'")
    ap.add_argument("--out", required=True)
    ap.add_argument("--sam", action="store_true", help="thêm biến thể <model>+sam (box từ mask SAM 2.1)")
    ap.add_argument("--sam-top", type=int, default=20)
    ap.add_argument("--low-thr", type=float, default=None)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshards", type=int, default=1)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--ckpt-every", type=int, default=50, help="lưu kết quả theo đợt mỗi N ảnh")
    ap.add_argument("--containers", action="store_true", help="lưu thêm box vật chứa để gộp món cùng đĩa (gdino)")
    ap.add_argument("--prompt-sets", default=None,
                    help='file JSON: list {"tag","mode":"fixed|class","prompts":[...],"extras":[...]} (chỉ gdino)')
    a = ap.parse_args()
    with open(a.items, encoding="utf-8") as f:
        items = json.load(f)
    prompt_sets = None
    if a.prompt_sets:
        with open(a.prompt_sets, encoding="utf-8") as f:
            prompt_sets = json.load(f)
    run(items, a.models.split(","), [p for p in a.prompts.split("|") if p.strip()], a.out, sam=a.sam,
        sam_top=a.sam_top, device=a.device, low_thr=a.low_thr, shard=a.shard, nshards=a.nshards,
        ckpt_every=a.ckpt_every, prompt_sets=prompt_sets, containers=a.containers)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile foodbox/viz.py
"""Vẽ box lên ảnh: mỗi class một màu, có nhãn chữ; ghép nhiều ảnh thành lưới."""
import colorsys

from PIL import Image, ImageDraw, ImageFont

GREEN, RED = (0, 200, 0), (230, 0, 0)


def color_for(i):
    """Màu cố định, dễ phân biệt cho class thứ i (bước tỉ lệ vàng trên vòng màu)."""
    r, g, b = colorsys.hsv_to_rgb((i * 0.618033988749895) % 1.0, 0.85, 0.95)
    return int(r * 255), int(g * 255), int(b * 255)


def _font(size):
    try:
        return ImageFont.load_default(size=size)  # Pillow >= 10.1
    except TypeError:
        return ImageFont.load_default()


def draw_boxes(img, boxes, labels=None, colors=None, width=None):
    """Trả bản sao ảnh đã vẽ box. labels/colors: list cùng độ dài với boxes (hoặc None)."""
    img = img.copy()
    d = ImageDraw.Draw(img)
    width = width or max(2, round(max(img.size) / 250))
    font = _font(max(12, round(max(img.size) / 40)))
    for k, b in enumerate(boxes):
        c = colors[k] if colors else RED
        x1, y1, x2, y2 = [float(v) for v in b[:4]]
        d.rectangle([x1, y1, x2, y2], outline=c, width=width)
        if labels and labels[k]:
            tb = d.textbbox((x1, y1), labels[k], font=font)
            ty = y1 - (tb[3] - tb[1]) - 4 if y1 - (tb[3] - tb[1]) - 4 > 0 else y1
            tb = d.textbbox((x1, ty), labels[k], font=font)
            d.rectangle([tb[0] - 2, tb[1] - 2, tb[2] + 2, tb[3] + 2], fill=c)
            d.text((x1, ty), labels[k], fill=(255, 255, 255), font=font)
    return img


def fit(img, max_side):
    """Thu nhỏ ảnh để cạnh dài = max_side (giữ tỉ lệ)."""
    s = max_side / max(img.size)
    return img if s >= 1 else img.resize((round(img.width * s), round(img.height * s)))


def titled(img, title, size=18):
    """Thêm dải tiêu đề phía trên ảnh."""
    font = _font(size)
    out = Image.new("RGB", (img.width, img.height + size + 8), (255, 255, 255))
    out.paste(img, (0, size + 8))
    ImageDraw.Draw(out).text((4, 2), title, fill=(0, 0, 0), font=font)
    return out


def grid(images, ncols, bg=(255, 255, 255)):
    """Ghép list ảnh thành lưới ncols cột (mỗi ô theo ảnh lớn nhất)."""
    if not images:
        return Image.new("RGB", (1, 1), bg)
    cw, ch = max(i.width for i in images), max(i.height for i in images)
    nrows = (len(images) + ncols - 1) // ncols
    out = Image.new("RGB", (cw * min(ncols, len(images)), ch * nrows), bg)
    for k, im in enumerate(images):
        out.paste(im, ((k % ncols) * cw, (k // ncols) * ch))
    return out

In [ ]:
# Tham số (giống bước 01 / 02 để dùng đúng các ảnh cũ)
CFG = dict(
    data_root="/kaggle/input",
    uec_class_ids=[1, 7, 11, 18, 23, 36, 87, 98, 206, 209], uec_per_class=20,
    food_n_classes=10, food_per_class=20,
    seed=0, timeout_min=45, out="/kaggle/working/output",
)

In [ ]:
# Dữ liệu: UEC (box thật) + Food-101 tiny, chọn giống hệt bước 01 / 02
import json, os, time
from foodbox import data
from foodbox.run_detect import launch, load_preds
OUT = CFG["out"]
for sub in ["logs", "preds_uec", "preds_food", "gt_labels"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
T0 = time.time()
uec_root = data.find_uec_root(CFG["data_root"])
assert uec_root, "Không tìm thấy UECFOOD-256"
cats = data.read_uec_categories(uec_root)
uec_items = data.sample_images(data.uec_class_images(uec_root, CFG["uec_class_ids"], cats), n_classes=None,
                               per_class=CFG["uec_per_class"], seed=CFG["seed"])
gts = data.convert_uec_to_yolo(uec_items, data.read_uec_boxes(uec_root),
                               {data.safe_name(cats[i]): i for i in CFG["uec_class_ids"]}, f"{OUT}/gt_labels")
food_roots = [r for r in data.find_class_roots(CFG["data_root"]) if not r.startswith(uec_root)]
food_items = data.dedupe_keys(data.sample_images(data.list_class_images(food_roots), None, CFG["food_n_classes"],
                                                 CFG["food_per_class"], CFG["seed"]))
for name, its in [("uec", uec_items), ("food", food_items)]:
    with open(f"{OUT}/items_{name}.json", "w", encoding="utf-8") as f:
        json.dump(its, f)
print(f"UEC {len(uec_items)} ảnh ({sum(len(g) >= 2 for g in gts.values())} ảnh nhiều món), Food-101 {len(food_items)} ảnh")

In [ ]:
# GPU 0 = UEC, GPU 1 = Food-101; mỗi ảnh 2 lượt (food + vật chứa), lưu theo đợt mỗi 50 ảnh
gpus = [0, 1] if torch.cuda.device_count() >= 2 else [0, 0]
jobs = [(g, ["--items", f"{OUT}/items_{name}.json", "--models", "gdino", "--containers", "--out", f"{OUT}/preds_{name}"],
         f"{OUT}/logs/detect_{name}.log") for g, name in zip(gpus, ["uec", "food"])]
if gpus[0] == gpus[1]:
    codes = [launch([j], timeout_min=CFG["timeout_min"], cwd="/tmp/foodbox_models")[0] for j in jobs]
else:
    codes = launch(jobs, timeout_min=CFG["timeout_min"], cwd="/tmp/foodbox_models")
for _, _, log in jobs:
    print(f"----- {log}")
    print("\n".join(open(log, encoding="utf-8", errors="replace").read().splitlines()[-5:]))
for name in ["uec", "food"]:
    P = load_preds(f"{OUT}/preds_{name}")["gdino"]["images"]
    n_c = sum(len(x.get("c_boxes", [])) > 0 for x in P.values())
    print(f"{name}: {len(P)} ảnh có kết quả, {n_c} ảnh có box vật chứa")
with open(f"{OUT}/summary.json", "w", encoding="utf-8") as f:
    json.dump({"config": CFG, "minutes": (time.time() - T0) / 60, "exit_codes": codes}, f, indent=1)
print(f"Xong sau {(time.time() - T0) / 60:.1f} phút")